# Helios Demo

Visualize soil moisture trajectories and local recommendation history.

In [ ]:
from pathlib import Path
import json
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
db_path = Path('../data/helios.db')
sample_path = Path('../data/sample_training_data.csv')

if db_path.exists():
    connection = sqlite3.connect(db_path)
    runs = pd.read_sql_query('SELECT * FROM prediction_runs ORDER BY created_at DESC LIMIT 10', connection)
    connection.close()
else:
    runs = pd.DataFrame()

if runs.empty and sample_path.exists():
    runs = pd.read_csv(sample_path).head(10)

runs.head()

In [ ]:
if 'response_json' in runs.columns and not runs.empty:
    latest = runs.iloc[0]
    payload = latest['response_json'] if isinstance(latest['response_json'], dict) else json.loads(latest['response_json'])
    forecast = payload['predicted_moisture']
    current_request = latest['request_json'] if isinstance(latest['request_json'], dict) else json.loads(latest['request_json'])
    current = current_request['soil_moisture_readings'][-1]['volumetric_water_content']
    chart_df = pd.DataFrame({
        'label': ['current', '24h', '48h', '72h'],
        'moisture': [current, forecast['moisture_24h'], forecast['moisture_48h'], forecast['moisture_72h']]
    })
    fig = px.line(chart_df, x='label', y='moisture', markers=True, title='Forecast Soil Moisture Trajectory')
    fig.show()
    payload
elif not runs.empty:
    preview = runs.iloc[0]
    chart_df = pd.DataFrame({
        'label': ['current', '24h', '48h', '72h'],
        'moisture': [preview['current_soil_moisture'], preview['target_moisture_24h'], preview['target_moisture_48h'], preview['target_moisture_72h']]
    })
    fig = px.line(chart_df, x='label', y='moisture', markers=True, title='Sample Moisture Trajectory')
    fig.show()
    chart_df
else:
    print('No stored runs or sample data found. Generate data or call the API first.')

In [ ]:
if 'decision' in runs.columns and not runs.empty:
    display_columns = ['created_at', 'field_id', 'decision', 'recommended_amount_mm', 'confidence_score']
    runs[display_columns]
else:
    numeric_cols = ['current_soil_moisture', 'target_moisture_24h', 'target_moisture_48h', 'target_moisture_72h']
    comparison = runs[numeric_cols].head(5).reset_index(drop=True)
    fig = go.Figure()
    for idx, row in comparison.iterrows():
        fig.add_trace(go.Scatter(x=['current', '24h', '48h', '72h'], y=row.tolist(), mode='lines+markers', name=f'scenario_{idx+1}'))
    fig.update_layout(title='Synthetic Scenario Comparison', yaxis_title='Volumetric Water Content')
    fig.show()